In [ ]:
from dijido import dawlla

dawlla.login()

In [ ]:
import os

# Get the path of the newest file in the "imports" directory

newest_file = max(
    (os.path.join("imports", f) for f in os.listdir("imports")),
    key=os.path.getctime,
)



In [ ]:
import json

if os.path.exists("legal_entity_map.json"):
    with open("legal_entity_map.json", "r") as f:
        legal_entity_map = json.load(f)
else:
    legal_entity_map = {}
    

In [ ]:
legal_entities = dawlla.get_legal_entities()

In [ ]:
import re
import xml.etree.ElementTree as ET

def parse_qbo_to_json(qbo_file_path):
    with open(qbo_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    # Skip all lines before the first XML-like tag
    xml_start_index = next(i for i, line in enumerate(lines) if line.strip().startswith('<'))
    content = ''.join(lines[xml_start_index:])

    # Fix SGML-style tags: convert <TAG>value into <TAG>value</TAG>
    # Only apply to lines with one tag and no closing
    fixed_lines = []
    for line in content.splitlines():
        line = line.strip()
        if re.match(r'^<[^/][^>]*>[^<]+$', line):
            tag = re.match(r'^<([^>]+)>', line).group(1)
            value = line[len(tag)+2:]
            fixed_line = f"<{tag}>{value}</{tag}>"
            fixed_lines.append(fixed_line)
        else:
            fixed_lines.append(line)

    fixed_content = '\n'.join(fixed_lines)

    # Now parse as XML
    root = ET.fromstring(fixed_content)

    # Extract STMTTRN entries
    stmttrns = root.findall('.//STMTTRN')
    transactions = []

    for stmt in stmttrns:
        txn = {}
        for child in stmt:
            txn[child.tag] = child.text.strip() if child.text else None
        transactions.append(txn)

    return transactions


In [ ]:
new_transactions = parse_qbo_to_json(newest_file)

In [ ]:
existing_transactions = dawlla.get_transactions()

In [ ]:
transaction_external_id_map = {}

for transaction in existing_transactions:

    if transaction["external_id"] is not None:
        transaction_external_id_map[transaction["external_id"]] = transaction

In [ ]:
for transaction in new_transactions:

    if transaction["FITID"] in transaction_external_id_map:
        # Transaction already exists, skip it
        continue

    # List the transaction and ask if the user wants to import it

    # Print it in a readable format
    print(f"Transaction ID: {transaction['FITID']}")
    print(f"Date: {transaction['DTPOSTED']}")
    print(f"Amount: {transaction['TRNAMT']}")
    print(f"Name: {transaction['NAME']}")

    print("Do you want to import this transaction? (y/n)")
    user_input = input().strip().lower()
    if user_input == 'y':

        legal_entity_name = transaction["NAME"]
        legal_entity_id = None

        if legal_entity_name in legal_entity_map:
            print(f"Legal entity exists in map: {legal_entity_map[legal_entity_name]}. Use? (y/n)")
            user_input = input().strip().lower()
            if user_input == 'y':
                legal_entity_id = legal_entity_map[legal_entity_name]
        
        if legal_entity_id is None:
            legal_entity_id = dawlla.create_legal_entity(legal_entity_name)
            legal_entity_map[legal_entity_name] = legal_entity_id

        transaction = {
            ""
        }

In [ ]:
result = dawlla.create_legal_entity({"name": "Bleger"})